Cleanup

In [2]:
import boto3
sagemaker = boto3.client('sagemaker')

In [3]:
# Deleting Endpoints

In [4]:
def list_all_endpoints(next_token = None):
    result = []
    resp = {}
    if next_token:
        resp = sagemaker.list_endpoints(NextToken=next_token)
    else:
        resp = sagemaker.list_endpoints() 
    resources = resp["Endpoints"]
    for resource in resources:
        result.append(resource)
    next_token = resp.get("NextToken")
    if next_token:
        return resources + list_all_endpoints(next_token)
    else:
        return resources

def delete_all_endpoints():
    for resource in list_all_endpoints():
        resource_name = resource["EndpointName"]
        print("Deleting endpoint {}".format(resource_name))
        # WARN: Are you sure you want do delete endpoints?
        # sagemaker.delete_endpoint(EndpointName=resource_name)
        
delete_all_endpoints()

Deleting endpoint mt-motor-anomaly-2020-05-13-16-17-37-246


# Deleting models


In [6]:
def get_models(next_token = None):
    result = []
    models_resp = {}
    if next_token:
        models_resp = sagemaker.list_models(NextToken=next_token)
    else:
        models_resp = sagemaker.list_models() 
    models = models_resp["Models"]
    for model in models:
        result.append(model)
    next_token = models_resp.get("NextToken")
    if next_token:
        return models + gen_models(next_token)
    else:
        return models

def delete_all_models():
    for model in get_models():
        model_name = model["ModelName"]
        print("Deleting model {}".format(model_name))
        # WARN: Are you sure you want do delete models?
        # sagemaker.delete_model(ModelName=model_name)
        
delete_all_models()

Deleting model mt-motor-anomaly-2020-05-13-16-17-37-246
Deleting model suntrack-ml-motor-forecast-rcf-2020-05-13-14-55-20-138
Deleting model mt-battery-deepar-2020-05-13-14-12-41-848
Deleting model mt-battery-deepar-2020-05-13-13-14-37-886
Deleting model mt-battery-deepar-2020-05-13-11-15-43-947
Deleting model mt-battery-deepar-2020-05-13-10-44-58-899
Deleting model randomcutforest-2020-04-08-11-16-44-015
Deleting model suntrack-battery-deepar-2020-04-08-11-09-30-101


In [ ]:
# Deleting S3 buckets prefixed with "moontracer"
s3 = boto3.client("s3")

def list_moontracer_buckets():
    response = s3.list_buckets()
    return [bucket["Name"] for bucket in response.get("Buckets", []) if bucket["Name"].startswith("moontracer") ]

def delete_bucket_contents(bucket_name):
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name):
        objects = page.get("Contents", [])
        if not objects:
            continue
        delete_items = [{"Key": obj["Key"]} for obj in objects]
        s3.delete_objects(Bucket=bucket_name, Delete={"Objects": delete_items})

def delete_moontracer_buckets():
    buckets = list_moontracer_buckets()
    for bucket_name in buckets:
        print(f"Deleting objects in bucket {bucket_name}")
        delete_bucket_contents(bucket_name)
        print(f"Deleting bucket {bucket_name}")
        # WARN: Are you sure you want to delete this bucket and all its contents?
        # s3.delete_bucket(Bucket=bucket_name)

delete_moontracer_buckets()


In [ ]:
# Delete CloudFormation stacks with names starting with "moontracer"
cfn = boto3.client("cloudformation")

def list_moontracer_stacks(next_token=None):
    stacks = []
    if next_token:
        resp = cfn.list_stacks(NextToken=next_token, StackStatusFilter=[
            "CREATE_IN_PROGRESS", "CREATE_FAILED", "CREATE_COMPLETE",
            "ROLLBACK_IN_PROGRESS", "ROLLBACK_FAILED", "ROLLBACK_COMPLETE",
            "DELETE_IN_PROGRESS", "DELETE_FAILED", "UPDATE_IN_PROGRESS",
            "UPDATE_COMPLETE_CLEANUP_IN_PROGRESS", "UPDATE_COMPLETE",
            "UPDATE_ROLLBACK_IN_PROGRESS", "UPDATE_ROLLBACK_FAILED",
            "UPDATE_ROLLBACK_COMPLETE_CLEANUP_IN_PROGRESS",
            "UPDATE_ROLLBACK_COMPLETE", "REVIEW_IN_PROGRESS",
            "IMPORT_IN_PROGRESS", "IMPORT_COMPLETE",
            "IMPORT_ROLLBACK_IN_PROGRESS", "IMPORT_ROLLBACK_FAILED",
            "IMPORT_ROLLBACK_COMPLETE"
        ] )
    else:
        resp = cfn.list_stacks(StackStatusFilter=[
            "CREATE_IN_PROGRESS", "CREATE_FAILED", "CREATE_COMPLETE",
            "ROLLBACK_IN_PROGRESS", "ROLLBACK_FAILED", "ROLLBACK_COMPLETE",
            "DELETE_IN_PROGRESS", "DELETE_FAILED", "UPDATE_IN_PROGRESS",
            "UPDATE_COMPLETE_CLEANUP_IN_PROGRESS", "UPDATE_COMPLETE",
            "UPDATE_ROLLBACK_IN_PROGRESS", "UPDATE_ROLLBACK_FAILED",
            "UPDATE_ROLLBACK_COMPLETE_CLEANUP_IN_PROGRESS",
            "UPDATE_ROLLBACK_COMPLETE", "REVIEW_IN_PROGRESS",
            "IMPORT_IN_PROGRESS", "IMPORT_COMPLETE",
            "IMPORT_ROLLBACK_IN_PROGRESS", "IMPORT_ROLLBACK_FAILED",
            "IMPORT_ROLLBACK_COMPLETE"
        ] )
    for stack in resp.get("StackSummaries", []):
        name = stack["StackName"]
        if name.startswith("moontracer"):
            stacks.append(name)
    next_token = resp.get("NextToken")
    if next_token:
        stacks.extend(list_moontracer_stacks(next_token))
    return stacks

def delete_moontracer_stacks():
    stacks = list_moontracer_stacks()
    for stack_name in stacks:
        print(f"Deleting CloudFormation stack {stack_name}")
        # WARN: Are you sure you want to delete this stack?
        # cfn.delete_stack(StackName=stack_name)

delete_moontracer_stacks()
